## Examining the Agent Loop

The agent loop is how Strands Agents process requests:

1. **Receive** user input
2. **Reason** using the LLM (decide what to do)
3. **Act** by calling a tool
4. **Observe** the tool result
5. **Repeat** or respond to the user

The table below shows each message in the loop — what the model said, which tool it called, and what result it received:

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()
console.print(f'Agent Loop Cycles: {advisor.event_loop_metrics.cycle_count}' if hasattr(agent, 'event_loop_metrics') else '')

# Get the last agent used in this notebook
active_agent = [v for v in dir() if not v.startswith('_')]

table = Table(title='Agent Messages', show_lines=True)
table.add_column('Role', style='green', width=10)
table.add_column('Text', style='magenta', max_width=50)
table.add_column('Tool', style='cyan', width=20)
table.add_column('Input', style='cyan', max_width=30)
table.add_column('Result', style='cyan', max_width=30)

for msg in advisor.messages[-6:]:  # Show last 6 messages for readability
    text = [c['text'] for c in msg['content'] if 'text' in c]
    tool_name = [c['toolUse']['name'] for c in msg['content'] if 'toolUse' in c]
    tool_input = [c['toolUse']['input'] for c in msg['content'] if 'toolUse' in c]
    tool_result = [c['toolResult']['content'][0] for c in msg['content'] if 'toolResult' in c]
    table.add_row(
        msg['role'],
        (text[-1][:100] + '...') if text and len(text[-1]) > 100 else (text[-1] if text else ''),
        tool_name[-1] if tool_name else '',
        (json.dumps(tool_input[-1])[:80] + '...') if tool_input else '',
        (json.dumps(tool_result[-1])[:80] + '...') if tool_result else '',
    )

console.print(table)


# Strands Agents with Bedrock AgentCore Memory — FSI Edition

This lab demonstrates how persistent memory transforms AI agents from stateless tools into context-aware advisors that remember client details across sessions.

## What You'll Build

1. Create an AgentCore Memory with three extraction strategies
2. Store rich FSI client conversations
3. Query each strategy independently and understand what it extracts
4. Build a memory-enabled agent that recalls client context
5. Demonstrate session handover — new TAM gets full client briefing

## Memory Strategies Explained

| Strategy | What It Extracts | FSI Example |
|----------|-----------------|-------------|
| **Summary** | Compressed session overview | "Discussed Acme Super's EKS migration timeline and latency requirements" |
| **User Preference** | Behavioral patterns & preferences | "Client prefers ESG investments, moderate risk appetite" |
| **Semantic** | Factual statements from user messages | "Acme Super spends $1.2M/month on AWS", "CTO is John Chen" |

## Setup

In [ ]:
import boto3

region = boto3.session.Session().region_name
NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'): NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'): NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'
print(f'Region: {region}, Model: {NOVA_PRO_MODEL_ID}')


## The Problem: Agents Forget Everything

Without persistent memory, every session starts blank:

In [ ]:
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are a financial advisor assistant. Be concise.',
)

# This will fail - agent has no memory of any client
agent('What is Acme Super\'s monthly AWS spend and who is their CTO?')


The agent gives a generic answer because it has **zero context**. Let's fix that.

---

## Step 1: Reset Memory (for clean demo)

Run this to delete any existing memory from previous runs:

In [ ]:
from bedrock_agentcore.memory import MemoryClient

memory_client = MemoryClient(region_name=region)
memories = memory_client.list_memories()

for m in memories:
    mid = m['id']
    if 'FSI' in mid:
        print(f'Deleting: {mid}')
        memory_client.delete_memory_and_wait(memory_id=mid)
        print(f'  ✅ Deleted')

print('Ready for fresh start')


## Step 2: Create Memory

We create a memory with three strategies. Each extracts different information from conversations.

⏱️ **Takes ~3 minutes to provision.**

In [ ]:
from bedrock_agentcore.memory.constants import StrategyType
from botocore.exceptions import ClientError

MEMORY_NAME = 'FSI_ClientMemory'
ACTOR_ID = 'tam_zohaib'

try:
    memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME,
        description='FSI client context memory. All extractions must be in English.',
        strategies=[
            {
                StrategyType.SUMMARY.value: {
                    'name': 'SessionSummary',
                    'description': 'Summarize sessions in English. Capture key decisions, action items, and client requirements.',
                    'namespaces': [f'fsi/{ACTOR_ID}/summaries/{{sessionId}}']
                }
            },
            {
                StrategyType.USER_PREFERENCE.value: {
                    'name': 'ClientPreferences',
                    'description': 'Extract client preferences in English. Focus on investment policy, risk appetite, technology preferences, and operational requirements.',
                    'namespaces': [f'fsi/{ACTOR_ID}/preferences']
                }
            },
            {
                StrategyType.SEMANTIC.value: {
                    'name': 'ClientFacts',
                    'description': 'Extract factual statements in English. Focus on spend figures, contacts, architecture details, dates, and requirements.',
                    'namespaces': [f'fsi/{ACTOR_ID}/facts/']
                }
            },
        ],
        event_expiry_days=30,
    )
    memory_id = memory.get('id')
    print(f'✅ Memory created: {memory_id}')
except ClientError as e:
    if 'already exists' in str(e):
        memories = memory_client.list_memories()
        memory_id = next(m['id'] for m in memories if MEMORY_NAME in m['id'])
        print(f'✅ Using existing: {memory_id}')
    else:
        raise e


## Step 3: Store Rich Client Conversations

We store detailed conversations about two FSI clients. The memory pipeline will automatically extract summaries, preferences, and facts.

In [ ]:
# === SESSION 1: Acme Super Onboarding ===
memory_client.create_event(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id='vanguard-001',
    messages=[
        ('I have been assigned Acme Super as my new client. They are a superannuation fund managing $220 billion in assets.', 'USER'),
        ('That is a significant account. What are their primary workloads on AWS?', 'ASSISTANT'),
        ('Their core trading platform runs on EC2 with Oracle on RDS. They want to migrate to EKS with Aurora PostgreSQL by Q3 2026. The CTO John Chen is sponsoring this migration. His email is john.chen@vanguard.com.au.', 'USER'),
        ('Noted the EKS migration target for Q3 2026, sponsored by CTO John Chen.', 'ASSISTANT'),
        ('Acme Super requires sub-10ms latency for trade execution in ap-southeast-2. They need 99.99 percent availability. Their DR site is us-west-2 with 15-minute RPO.', 'USER'),
        ('Critical NFRs captured: sub-10ms latency, 99.99% availability, DR in us-west-2 with 15-min RPO.', 'ASSISTANT'),
        ('Their investment policy is ESG-only. They refuse any exposure to fossil fuels, gambling, or weapons. They report to APRA quarterly. Their risk appetite is moderate.', 'USER'),
        ('ESG-only policy noted with APRA quarterly reporting and moderate risk appetite.', 'ASSISTANT'),
        ('Acme Super currently spends $1.2 million per month on AWS. EC2 is 45 percent of spend, RDS is 25 percent. They have zero Reserved Instances which is a big optimization opportunity.', 'USER'),
        ('Significant RI/SP opportunity on $1.2M monthly spend.', 'ASSISTANT'),
        ('The trading platform handles 50000 transactions per second at peak. They use Kafka for streaming and Redis for caching. The backup contact is Sarah Liu, Head of Platform Engineering.', 'USER'),
        ('Architecture: 50K TPS, Kafka, Redis. Backup contact: Sarah Liu (Head of Platform Eng).', 'ASSISTANT'),
    ],
)
print('✅ Session 1: Acme Super onboarding stored')

# === SESSION 2: Z-Pay Onboarding ===
memory_client.create_event(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id='afterpay-001',
    messages=[
        ('My other client Z-Pay is a BNPL fintech. They process 5 million transactions daily and need real-time fraud detection under 100ms. Their fraud engine runs on SageMaker with custom models.', 'USER'),
        ('Z-Pay: 5M daily transactions, sub-100ms fraud detection on SageMaker.', 'ASSISTANT'),
        ('Z-Pay is very worried about upcoming ASIC regulations on BNPL. Their legal team requires all transaction data stays in Australia. Nothing can leave ap-southeast-2.', 'USER'),
        ('Data sovereignty: ap-southeast-2 only. ASIC regulatory concern noted.', 'ASSISTANT'),
        ('They spend $800K per month on AWS. They prefer Graviton instances for cost savings. DR is in us-west-2 with 5-minute RPO. The main contact is David Park, VP Engineering, david.park@afterpay.com.', 'USER'),
        ('Z-Pay: $800K/month, Graviton preference, DR us-west-2 (5-min RPO), contact David Park.', 'ASSISTANT'),
    ],
)
print('✅ Session 2: Z-Pay onboarding stored')
print()
print('⏱️ Waiting 30 seconds for memory pipeline to process...')

import time
time.sleep(30)
print('✅ Ready to query')


## Step 4: Query Each Memory Strategy

### Summary Strategy
Returns compressed session overviews. Best for: *"What did we discuss last time?"*

In [ ]:
print('📋 SUMMARY STRATEGY')
print('   Query: "Acme Super trading platform and migration"')
print('-' * 60)

results = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi/{ACTOR_ID}/summaries/vanguard-001',
    query='Acme Super trading platform and migration',
    top_k=2
)

if results:
    for r in results:
        print(f'\n  Score: {r["score"]:.2f}')
        print(f'  {r["content"]["text"][:500]}')
else:
    print('  (No summaries yet - may need more processing time)')


### User Preference Strategy
Extracts preferences and behavioral patterns. Best for: *"What does this client prefer?"*

In [ ]:
print('💡 USER PREFERENCE STRATEGY')
print('   Query: "investment policy and risk appetite"')
print('-' * 60)

results = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi/{ACTOR_ID}/preferences',
    query='investment policy and risk appetite',
    top_k=5
)

for r in results:
    print(f'\n  Score: {r["score"]:.2f}')
    print(f'  {r["content"]["text"][:300]}')


### Semantic Strategy
Extracts factual statements from **user messages only**. Best for: *"What are the hard facts?"*

⚠️ Only facts stated by the USER are stored — not assistant responses.

In [ ]:
print('🧠 SEMANTIC STRATEGY')
print('   Query: "Acme Super AWS spend contacts architecture"')
print('-' * 60)

results = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi/{ACTOR_ID}/facts/',
    query='Acme Super AWS spend contacts architecture',
    top_k=7
)

for r in results:
    print(f'\n  Score: {r["score"]:.2f}')
    print(f'  {r["content"]["text"]}')


---

## Step 5: Memory-Enabled Agent

Now let's build an agent that automatically queries memory before responding:

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def recall_client(query: str) -> str:
    '''Retrieve stored facts and preferences about a client.
    Args:
        query: What to recall (e.g., "Acme Super contacts" or "Z-Pay latency")
    '''
    all_results = []
    for ns in [f'fsi/{ACTOR_ID}/preferences', f'fsi/{ACTOR_ID}/facts/']:
        results = memory_client.retrieve_memories(
            memory_id=memory_id, namespace=ns, query=query, top_k=5
        )
        all_results.extend([r['content']['text'] for r in results if r['score'] > 0.3])
    return '\n'.join(all_results[:8]) if all_results else 'No context found.'

advisor = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are an FSI advisor. Always recall client context before answering. Cite specific facts.',
    tools=[recall_client],
)

print('✅ Memory-enabled agent ready')


In [ ]:
# Use Case 1: Specific client question
advisor('What is Acme Super\'s monthly AWS spend and what optimization opportunities exist?')


In [ ]:
# Use Case 2: Cross-client comparison
advisor('Compare the DR strategies and RPO targets for Acme Super vs Z-Pay.')


---

## Step 6: Session Handover

The most powerful FSI use case: a **new advisor takes over** the account with zero prior context. The agent provides a full briefing from memory.

In [ ]:
# Simulate: completely new session, new advisor, no conversation history
new_advisor = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are helping a new advisor prepare for their first client meeting. Provide a comprehensive briefing using stored context.',
    tools=[recall_client],
)

new_advisor('I just took over the Acme Super account. Brief me on everything: contacts, architecture, requirements, spend, migration plans, and any risks.')


In [ ]:
# New advisor asks about the other client too
new_advisor('What about Z-Pay? What are their key concerns and who do I contact?')


---

## Cleanup (Optional)

In [ ]:
# Delete memory when done
# memory_client.delete_memory_and_wait(memory_id=memory_id)
# print('✅ Memory deleted')


## Summary

| What We Did | FSI Value |
|------------|----------|
| Stored client conversations | Build institutional knowledge |
| Summary strategy | Quick session recaps for follow-ups |
| Preference strategy | Personalized recommendations (ESG, risk) |
| Semantic strategy | Hard facts recall (spend, contacts, dates) |
| Memory-enabled agent | Context-aware responses without re-asking |
| Session handover | New advisor gets full briefing instantly |

### Key Insight

Memory turns a stateless AI tool into a **relationship-aware advisor** that accumulates knowledge over time — exactly what FSI clients expect from their support team.